In [2]:
### Libraries
from trino.dbapi import connect
import pandas as pd,time
conn = connect(
    host = "202.4.114.56",
    port = "8888",
    user = "gazu",
    catalog= "padakhep",
    schema= "gold",
)
cur = conn.cursor()

def query_to_df(sql):
    cur.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

In [3]:
query_to_df("SELECT * from loans LIMIT 100")

,balance,created_at,created_by,cutoff_next_payable_date,daily_interest,disbursed_date,holiday_count,id,inactive_date,is_active,...,loan_condition,loan_product_id,member_id,next_unpaid_schedule_id,overdue,recovery_method,status,total_daily_interest,total_interest,updated_at
0,102600.0,2026-06-30 16:10:50+00:00,398,None,None,2026-06-30,0,2258065,None,Yes,...,regular,25,1015706,87734107,0,flat,active,None,12350.00,2026-07-21 12:45:59+00:00
1,201052.0,2026-07-16 10:24:07+00:00,7240,None,1052,2026-07-16,0,2270884,None,Yes,...,regular,194,1221689,87945280,0,amortization,active,1052,1052.00,2026-07-23 18:05:29+00:00
2,340500.0,2026-07-15 09:13:58+00:00,5325,None,None,2026-07-15,0,2268576,None,Yes,...,regular,22,1354689,87909899,0,flat,active,None,40500.00,2026-07-15 09:13:58+00:00
3,113500.0,2026-07-13 11:58:14+00:00,2110,None,None,2026-07-13,0,2265222,None,Yes,...,regular,41,1170926,87856545,0,flat,active,None,13500.00,2026-07-13 11:58:14+00:00
4,220289.0,2026-07-22 14:10:30+00:00,2895,None,289,2026-07-22,0,2280697,None,Yes,...,regular,193,1052422,88097016,0,amortization,active,289,289.00,2026-07-23 18:18:43+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,55494.0,2026-06-23 13:59:06+00:00,6498,None,326,2026-06-23,0,2245992,None,Yes,...,regular,194,1352525,87491075,0,amortization,active,1194,1194.00,2026-07-23 18:00:29+00:00
96,90800.0,2026-07-22 15:24:18+00:00,7111,None,None,2026-07-22,0,2281595,None,Yes,...,regular,29,1365085,88111161,0,flat,active,None,10800.00,2026-07-22 15:24:18+00:00
97,33000.0,2026-06-25 10:14:38+00:00,1355,None,0,2026-06-25,0,2251152,None,Yes,...,regular,199,988062,87580416,0,amortization,active,0,0.00,2026-07-23 18:42:34+00:00
98,170250.0,2026-06-25 13:15:16+00:00,6135,None,None,2026-06-25,0,2252182,None,Yes,...,regular,48,175969,87594534,0,flat,active,None,20250.00,2026-06-25 13:15:16+00:00


In [5]:
end_date = '2026-07-01'
query_to_df(f"""SELECT count(*) from loans where next_unpaid_schedule_id = 0 and is_active = 'Yes' and disbursed_date < DATE '{end_date}'""")

,_col0
0,163599


In [3]:
#start_date = '2025-01-01'
end_date = '2026-07-01'
loan_limit = 300000
batch_size = 20000

ranked_sql = f"""
     WITH selected_loans AS (
      SELECT loans.id AS loan_id
      FROM loans
      WHERE
      -- loans.inactive_date >= DATE 'start_date'
        -- loans.inactive_date < DATE 'end_date'
        next_unpaid_schedule_id = 0
        and loans.is_active = 'Yes'
        AND loans.disbursed_date < DATE '{end_date}'
        AND  (
        loans.recovery_method = 'flat'
        OR loans.recovery_method IS NULL
         )
      ORDER BY loans.disbursed_date DESC , loans.id DESC -- add later to remove duplicate
      LIMIT 100
    )
    SELECT
      loans.member_id,
      loans.id,
      loans.is_active,
      loans.next_unpaid_schedule_id,
       ls.id AS schedule_id,
      ROW_NUMBER() OVER (PARTITION BY loans.id ORDER BY ls.id) AS payment_num
    FROM loans
    JOIN selected_loans sl ON sl.loan_id = loans.id
    LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id
    WHERE ls.is_holiday = 'False'
"""
ranked_df = query_to_df(ranked_sql)
max_payments = int(ranked_df['payment_num'].max())

case_columns = ",\n      ".join(
    f'MAX(CASE WHEN rn.payment_num = {n} THEN rn.paid_pattern END) AS "installment_{n}"'
    for n in range(1, max_payments + 1)
)

# Step 2: pull loans in batches, using OFFSET/LIMIT on the same ordered selected_loans set
all_batches = []
offset = 0

while offset < loan_limit:
    this_batch_size = min(batch_size, loan_limit - offset)

    final_sql = f"""
        WITH selected_loans AS (
          SELECT loans.id AS loan_id
          FROM loans
          WHERE
          -- loans.id = 1861037 and
           (
            loans.recovery_method = 'flat'
            OR loans.recovery_method IS NULL
        )  and
            -- loans.inactive_date >= DATE 'start_date' AND
           --  loans.inactive_date < DATE 'end_date'
            next_unpaid_schedule_id = 0
            and loans.is_active = 'Yes'
            AND loans.disbursed_date < DATE '{end_date}'
          ORDER BY loans.disbursed_date DESC, loans.id DESC   -- add later to remove duplicate
          OFFSET {offset} LIMIT {this_batch_size}
        ),
        ranked AS (
          SELECT
            m.name,
            m.gender, m.dob AS dateofbirth,
            m.current_addr_district,
            m.current_addr_thana,
            m.admission_date AS admission_date,
            m.occupation AS occupation,
            m.occupation_alt alternative_occupation,
            m.marital_status AS maritial_status,
            m.education AS education,
            m.total_family_member,
            m.total_earning_member,
            m.total_monthly_income,
            m.yearly_total_income,
            m.yearly_total_expenses,
            m.yearly_total_savings,
            m.home_owner_type,
            m.year_of_rent,
            m.farm_land_amt,
            khaats.name loan_use_sources,
            proposer.old_id AS proposer_oldid,
            approver.old_id AS approvers_id,
            designations.designation AS approvers_designation,
            loans.id AS loan_id,
            loans.disbursed_date,
            loans.inactive_date,
            loans.loan_amount AS loan_amount,
            loans.next_unpaid_schedule_id,
            MIN(ls.due_date) OVER (PARTITION BY loans.id) AS schedule_start_date,
            MAX(ls.due_date) OVER (PARTITION BY loans.id) AS schedule_end_date,
            MAX(ls.kisti_paid_date) OVER (PARTITION BY loans.id) AS last_paid_date,
            MAX(
              CASE
                WHEN kisti_paid_date > due_date THEN date_diff('day', due_date, kisti_paid_date)
                ELSE 0
              END
            ) OVER (PARTITION BY loans.id) AS schedule_max_days_late,
            loans.member_id,
            ls.id AS schedule_id,
            loan_products.inst_type AS installmenttype,
            loan_products.inst_year AS installmentyear,
            loan_products.inst_total AS totalInstallmentCount,
            loan_applications.dofa_no AS loan_cycle_no,
            loan_applications.applied_loan_amount AS applied_loan_amount,
            loan_components.name AS loan_component,
            loan_components.loan_type as loan_component_type,
            -- COALESCE(count(tl.id), 0) transaction_count,
            CASE
              WHEN due_date = kisti_paid_date AND kisti_amount = paid_amount THEN 'regular'
              WHEN due_date > kisti_paid_date AND kisti_amount = paid_amount THEN 'advanced'
              ELSE 'bokeya'
            END AS paid_pattern,
            ROW_NUMBER() OVER (PARTITION BY loans.id ORDER BY ls.id) AS payment_num
          FROM loans
          JOIN selected_loans sl ON sl.loan_id = loans.id
          LEFT JOIN loan_schedules ls ON ls.loan_id = loans.id and ls.is_holiday = 'False'
          LEFT JOIN members m ON m.id = loans.member_id
          LEFT JOIN loan_products ON loan_products.id = loans.loan_product_id
          LEFT JOIN loan_applications ON loan_applications.id = loans.loan_application_id
          LEFT JOIN loan_components ON loan_components.id = loan_products.component_id
        LEFT JOIN designations ON designations.id = loan_applications.approver_designation_id
        LEFT JOIN employees approver ON approver.id = loan_applications.approved_by
        LEFT JOIN employees proposer ON proposer.id = loan_applications.created_by
        LEFT JOIN Khaats on CAST(khaats.id AS VARCHAR) = loan_applications.loan_khaat
        --   LEFT JOIN transactions_loans tl ON tl.loan_schedule_id = ls.id
        WHERE ls.is_holiday = 'False'
        ),
    txn_counts AS (
      SELECT
        ls.loan_id,
        COUNT(tl.id) AS transaction_count,
        count(distinct date_of_action) paymentdaycount
      FROM loan_schedules ls
      JOIN selected_loans sl ON sl.loan_id = ls.loan_id
      LEFT JOIN transactions_loans tl ON tl.loan_schedule_id = ls.id
       where ls.is_holiday = 'False'
      GROUP BY ls.loan_id
    ),

        loan_level AS (
      SELECT DISTINCT
        l.id AS loan_id,
        l.member_id,
        l.loan_amount,
        l.inactive_date,
        loan_applications.dofa_no
      FROM loans l
      LEFT JOIN loan_applications ON loan_applications.id = l.loan_application_id
        WHERE l.member_id IN (
    SELECT DISTINCT l2.member_id
    FROM loans l2
    JOIN selected_loans sl ON sl.loan_id = l2.id
  )
    ),
    loan_level_lag AS (
  SELECT
    loan_id,
       LAG(inactive_date) OVER (PARTITION BY member_id ORDER BY dofa_no) AS prev_loan_inactive_date,
        LAG(loan_amount) OVER (PARTITION BY member_id ORDER BY dofa_no) AS prev_loan_amount,
        LAG(dofa_no) OVER (PARTITION BY member_id ORDER BY dofa_no) AS prev_cycle_no,
        AVG(loan_amount) OVER (partition BY member_id ORDER BY dofa_no ROWS Between 3 Preceding and 1 Preceding ) AS last3_avg_loan_amount
      FROM loan_level
    ),

    max_savings_withdrawal AS (
  SELECT
    sl.loan_id as lo_id,
    Count(ts.id) total_savings_transactions,
    SUM(CASE WHEN ts.transaction_type <> 'withdraw' THEN ts.amount ELSE 0 END) AS deposit_amount,
SUM(CASE WHEN ts.transaction_type = 'withdraw' THEN ts.amount ELSE 0 END) AS refund_amount
  FROM selected_loans sl
  JOIN loans l ON l.id = sl.loan_id
  JOIN transactions_savings ts
    ON ts.member_id = l.member_id
    AND ts.date_of_action >= l.disbursed_date
    AND ts.date_of_action < l.inactive_date
    AND ts.savings_type = 'rspw'
  GROUP BY sl.loan_id
)

        SELECT
    -- Member Information
    rn.member_id,
    rn.name,

    -- Personal Information
    rn.dateofbirth,
    rn.gender,
    rn.current_addr_district,
    rn.current_addr_thana,
    rn.maritial_status,
    rn.admission_date,

    -- Education & Occupation
    rn.education,
    rn.occupation,
    rn.alternative_occupation,

    -- Family & Financial Information
    rn.total_family_member,
    rn.total_earning_member,
    rn.total_monthly_income,
    rn.yearly_total_income,
    rn.yearly_total_expenses,
    rn.yearly_total_savings,

    -- Housing & Assets
    rn.home_owner_type,
    rn.year_of_rent,
    rn.farm_land_amt,

    -- Loan Information

    ll.prev_cycle_no,
    ll.prev_loan_amount,
    rn.loan_id,
    rn.approvers_designation,
    rn.proposer_oldid,
    rn.approvers_id,
    rn.loan_cycle_no,
    rn.loan_component,
    rn.loan_component_type,
    rn.loan_use_sources,
    rn.applied_loan_amount,
    rn.loan_amount,

    ll.last3_avg_loan_amount,
    -- Loan Schedule & Repayment
    ll.prev_loan_inactive_date,
    rn.disbursed_date,
    rn.inactive_date,
    rn.schedule_start_date,
    rn.schedule_end_date,
    rn.last_paid_date,
    rn.installmenttype,
    rn.installmentyear,
    rn.totalInstallmentcount,
    tc.transaction_count,
    tc.paymentdaycount,
    rn.schedule_max_days_late,
    total_savings_transactions,
    refund_amount,
    deposit_amount,
    next_unpaid_schedule_id,

    -- Calculated / Case Information
    {case_columns}

FROM ranked rn
 LEFT JOIN txn_counts tc ON tc.loan_id = rn.loan_id
LEFT JOIN loan_level_lag ll ON ll.loan_id = rn.loan_id
LEFT JOIN max_savings_withdrawal on max_savings_withdrawal.lo_id = rn.loan_id

GROUP BY
    rn.member_id,
    rn.name,
    rn.dateofbirth,
    rn.gender,
    rn.maritial_status,
    rn.admission_date,
    rn.education,
    rn.occupation,
    rn.alternative_occupation,
    rn.total_family_member,
    rn.total_earning_member,
    rn.total_monthly_income,
    rn.yearly_total_income,
    rn.yearly_total_expenses,
    rn.yearly_total_savings,
    rn.home_owner_type,
    rn.year_of_rent,
    rn.farm_land_amt,
    rn.loan_id,
    rn.loan_cycle_No,
    rn.loan_component,
    rn.loan_component_type,
    rn.loan_use_sources,
    rn.loan_amount,
    rn.disbursed_date,
    rn.inactive_date,
    rn.schedule_start_date,
    rn.schedule_end_date,
    rn.installmenttype,
    rn.installmentyear,
    rn.totalInstallmentcount,
    rn.last_paid_date,
    rn.applied_loan_amount,
    rn.current_addr_district,
     rn.schedule_max_days_late,
    rn.current_addr_thana,
    rn.approvers_designation,
    rn.proposer_oldid,
    rn.approvers_id,
    ll.prev_loan_inactive_date,
    ll.prev_loan_amount,
    ll.prev_cycle_no,
    ll.last3_avg_loan_amount,
    tc.transaction_count,
    tc.paymentdaycount,
    refund_amount,
     deposit_amount,
     total_savings_transactions, next_unpaid_schedule_id


ORDER BY
    rn.disbursed_date DESC
    """

    try:
        batch_df = query_to_df(final_sql)
    except Exception as e:
        print(f"Batch failed at offset {offset}: {e}")
        break

    if batch_df.empty:
        break
    batch_df.head()
    all_batches.append(batch_df)
    print(f"Fetched offset {offset}–{offset + len(batch_df)} ({len(batch_df)} rows)")

    offset += this_batch_size
    time.sleep(1)   # small pause between batches, easy on the server


result_df = pd.concat(all_batches, ignore_index=True) if all_batches else pd.DataFrame()
result_df

Fetched offset 0–20000 (20000 rows)
Fetched offset 20000–40000 (20000 rows)
Fetched offset 40000–60000 (20000 rows)
Fetched offset 60000–80000 (20000 rows)
Fetched offset 80000–100000 (20000 rows)
Fetched offset 100000–120000 (20000 rows)
Fetched offset 120000–140000 (20000 rows)
Fetched offset 140000–160000 (20000 rows)
Fetched offset 160000–163592 (3592 rows)


,member_id,name,dateofbirth,gender,current_addr_district,current_addr_thana,maritial_status,admission_date,education,occupation,...,installment_37,installment_38,installment_39,installment_40,installment_41,installment_42,installment_43,installment_44,installment_45,installment_46
0,345491,Razes Day,1991-01-01,Male,Rangamati,Rajosthali,Unmarried,2016-01-18,Pre-Primary,Business,...,None,None,None,None,None,None,None,None,None,None
1,880960,Rina Khatun,2000-03-03,Female,Dinajpur,Dinajpur,Married,2024-02-12,Pre-Primary,Housewife,...,None,None,None,None,None,None,None,None,None,None
2,250170,Minoti,1993-05-09,Female,Dinajpur,Birganj,Married,2021-06-21,Pre-Primary,Housewife,...,None,None,None,None,None,None,None,None,None,None
3,474809,Rukhsana Khatun,1983-06-08,Female,Dinajpur,Dinajpur,Married,2022-01-13,Primary,Service,...,None,None,None,None,None,None,None,None,None,None
4,369720,MstShahinur Begum,1985-06-12,Female,Dinajpur,Birampur,Married,2012-11-19,Pre-Primary,Housewife,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163587,23662,Nurunnahar Begam,1988-06-17,Female,Gazipur,Kaligonj,Married,2015-12-01,PSC,Housewife,...,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya
163588,172048,Mst. Mahfuza,1982-10-08,Female,Rajshahi,Mohanpur,Married,2015-11-29,PSC,Housewife,...,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya
163589,23888,Shela Akter,1985-12-11,Female,Gazipur,Kaligonj,Married,2014-04-10,JSC,Housewife,...,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya
163590,415243,Rukeya Begum,1976-05-23,Female,Coxs Bazar,Chakoria,Married,2018-04-13,Primary,Housewife,...,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya,bokeya


In [9]:
result_df.to_csv("overdue_loanee_dataset.csv", index=False, encoding="utf-8-sig")